# Autoencoder + PCA preprocessing for Quantum ML

This notebook loads the `classical_qi_predictions_all_splits` dataset, applies PCA and an autoencoder to reduce dimensionality, and saves the transformed embeddings for downstream Quantum ML experiments.

## Dataset

The notebook expects the dataset at `outputs/classical_qi_predictions_all_splits.csv` relative to this notebook's folder.

In [19]:
# Imports and environment checks
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import torch
from torch import nn

print('pandas', pd.__version__)
print('numpy', np.__version__)
print('sklearn available')
print('torch', torch.__version__)

pandas 2.3.3
numpy 1.26.4
sklearn available
torch 2.8.0


In [20]:
# Paths and loading
base = Path('.')  # notebook directory when opened in Jupyter/Lab
data_path = base / 'outputs' / 'classical_qi_predictions_all_splits.csv'
assert data_path.exists(), f'Data not found: {data_path}'
df = pd.read_csv(data_path)
print('Loaded', data_path)
display(df)

Loaded outputs/classical_qi_predictions_all_splits.csv


,family,split,scenario,dataset_name,true_dataset_label,max_source_probability,mean_source_probability,predicted_source_count,predicted_dataset_label,predicted_source_ips,shortlisted_source_count
0,family_a,test,attack,attack_test_01,1,0.995904,0.699914,89,1,10.46.195.235;10.244.108.70;10.43.75.186;10.31...,89
1,family_a,test,attack,attack_test_02,1,0.995336,0.658222,71,1,10.14.8.78;10.101.104.90;10.11.73.41;10.146.58...,71
2,family_a,test,normal,normal_test_01,0,0.279839,0.050306,0,0,NaN,0
3,family_a,test,normal,normal_test_02,0,0.227024,0.042665,0,0,NaN,0
4,family_a,train,attack,attack_train_01,1,0.996542,0.629347,59,1,10.22.83.214;10.36.16.224;10.184.166.178;10.37...,59
5,family_a,train,attack,attack_train_02,1,0.995201,0.656432,71,1,10.185.147.26;10.59.46.74;10.194.118.196;10.68...,71
6,family_a,train,attack,attack_train_03,1,0.995316,0.656858,69,1,10.14.74.245;10.132.255.139;10.58.164.90;10.11...,69
7,family_a,train,attack,attack_train_04,1,0.995361,0.622197,58,1,10.193.24.125;10.100.22.103;10.179.61.14;10.14...,58
8,family_a,train,attack,attack_train_05,1,0.996610,0.677937,74,1,10.150.167.164;10.77.154.8;10.15.70.90;10.213....,74
9,family_a,train,attack,attack_train_06,1,0.995900,0.657572,74,1,10.250.159.192;10.226.109.198;10.54.167.11;10....,74


In [21]:
# Quick inspection and feature selection
print('shape:', df.shape)
print('columns:', list(df.columns))
# Select numeric columns only as features (drop identifiers / non-numeric)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print('numeric columns count:', len(numeric_cols))
X = df[numeric_cols].copy()
# Simple NaN handling: fill or drop depending on case (here fill with mean)
X = X.fillna(X.mean())
X.head()

shape: (48, 11)
columns: ['family', 'split', 'scenario', 'dataset_name', 'true_dataset_label', 'max_source_probability', 'mean_source_probability', 'predicted_source_count', 'predicted_dataset_label', 'predicted_source_ips', 'shortlisted_source_count']
numeric columns count: 6


,true_dataset_label,max_source_probability,mean_source_probability,predicted_source_count,predicted_dataset_label,shortlisted_source_count
0,1,0.995904,0.699914,89,1,89
1,1,0.995336,0.658222,71,1,71
2,0,0.279839,0.050306,0,0,0
3,0,0.227024,0.042665,0,0,0
4,1,0.996542,0.629347,59,1,59


In [22]:
# Autoencoder followed by PCA
# This approach uses a non-linear autoencoder to learn a compressed representation, 
# and then PCA is applied to the latent space for further linear dimensionality reduction.

# Define the Autoencoder architecture
class Autoencoder(nn.Module):
    def __init__(self, input_dim, encoding_dim):
        super(Autoencoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, encoding_dim)
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim),
            nn.Sigmoid()  # To produce output in [0, 1]
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Normalize the data to be in the range [0, 1] for the Sigmoid output
scaler_01 = MinMaxScaler()
X_scaled_01 = scaler_01.fit_transform(X)
X_tensor = torch.tensor(X_scaled_01, dtype=torch.float32)

# Hyperparameters
input_dim = X.shape[1]
encoding_dim = 32  # Intermediate latent space dimension
final_latent_dim = 10
epochs = 1000
batch_size = 64
learning_rate = 1e-3

# Model, loss, and optimizer
autoencoder = Autoencoder(input_dim, encoding_dim)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=learning_rate)

# Training loop
for epoch in range(epochs):
    for i in range(0, len(X_tensor), batch_size):
        batch = X_tensor[i:i+batch_size]
        # Forward pass
        outputs = autoencoder(batch)
        loss = criterion(outputs, batch)
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# Get the latent space representation from the encoder
with torch.no_grad():
    latent_representation = autoencoder.encoder(X_tensor).numpy()

# Apply PCA on the latent space
pca_on_latent = PCA(n_components=final_latent_dim)
final_representation = pca_on_latent.fit_transform(latent_representation)

print("Shape of the final representation:", final_representation.shape)

# Save the final representation
df_final = pd.DataFrame(final_representation, columns=[f'latent_{i+1}' for i in range(final_latent_dim)])
output_path = base / 'outputs' / 'autoencoder_pca_representation.csv'
df_final.to_csv(output_path, index=False)
print(f"Saved final representation to {output_path}")

Epoch [1/1000], Loss: 0.2099
Epoch [2/1000], Loss: 0.2087
Epoch [3/1000], Loss: 0.2076
Epoch [4/1000], Loss: 0.2064
Epoch [5/1000], Loss: 0.2052
Epoch [6/1000], Loss: 0.2040
Epoch [7/1000], Loss: 0.2027
Epoch [8/1000], Loss: 0.2014
Epoch [9/1000], Loss: 0.2000
Epoch [10/1000], Loss: 0.1984
Epoch [11/1000], Loss: 0.1968
Epoch [12/1000], Loss: 0.1949
Epoch [13/1000], Loss: 0.1929
Epoch [14/1000], Loss: 0.1907
Epoch [15/1000], Loss: 0.1882
Epoch [16/1000], Loss: 0.1853
Epoch [17/1000], Loss: 0.1821
Epoch [18/1000], Loss: 0.1785
Epoch [19/1000], Loss: 0.1745
Epoch [20/1000], Loss: 0.1700
Epoch [21/1000], Loss: 0.1650
Epoch [22/1000], Loss: 0.1595
Epoch [23/1000], Loss: 0.1534
Epoch [24/1000], Loss: 0.1470
Epoch [25/1000], Loss: 0.1401
Epoch [26/1000], Loss: 0.1329
Epoch [27/1000], Loss: 0.1257
Epoch [28/1000], Loss: 0.1185
Epoch [29/1000], Loss: 0.1114
Epoch [30/1000], Loss: 0.1046
Epoch [31/1000], Loss: 0.0980
Epoch [32/1000], Loss: 0.0914
Epoch [33/1000], Loss: 0.0848
Epoch [34/1000], Lo

In [23]:
display(df_final)

,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,latent_7,latent_8,latent_9,latent_10
0,9.368065,1.036052,0.010720,-0.041680,0.001253,0.001062,-0.000434,-1.276143e-04,-0.000160,0.000084
1,8.467204,0.600399,-0.009791,0.002493,-0.000218,-0.001551,0.000525,1.302225e-04,0.000057,-0.000033
2,-5.503292,0.207655,0.102274,0.012478,0.008898,0.000218,0.000171,-5.027790e-05,-0.000109,-0.000181
3,-5.778824,0.196707,0.078323,0.005727,0.001629,-0.000786,-0.000296,2.176666e-05,0.000036,0.000044
4,7.883273,0.311859,-0.022694,0.030746,-0.000652,0.000514,-0.000213,-1.113751e-04,-0.000030,0.000021
5,8.460984,0.600989,-0.009232,0.001479,-0.000205,-0.001607,0.000548,1.497648e-04,0.000062,-0.000036
6,8.378444,0.551410,-0.013011,0.009120,-0.000372,-0.001382,0.000498,1.296817e-04,0.000094,-0.000052
7,7.813932,0.290467,-0.022145,0.030163,-0.000678,0.000766,-0.000187,1.618354e-05,0.000083,-0.000043
8,8.662682,0.668558,-0.010394,0.002413,-0.000055,-0.000759,0.000191,-6.814333e-05,-0.000037,0.000021
9,8.595797,0.674299,-0.004153,-0.008890,0.000108,-0.001588,0.000426,5.281822e-05,-0.000082,0.000042


In [ ]:
import matplotlib.pyplot as plt

# 1. Check PCA Explained Variance
explained_variance = sum(pca_on_latent.explained_variance_ratio_)
print(f"Total PCA Explained Variance ({final_latent_dim} components): {explained_variance:.4f}")
print("If this is close to 1.0, the 10 components captured almost all the variance of the 32D latent space.\n")

# 2. Visualize the top 2 components of the latent space
plt.figure(figsize=(10, 6))

if 'scenario' in df.columns:
    # Plot with clusters colored by the 'scenario' label (attack vs normal)
    labels = df['scenario'].unique()
    for label in labels:
        idx = df['scenario'] == label
        plt.scatter(
            final_representation[idx, 0], 
            final_representation[idx, 1], 
            label=label, 
            alpha=0.5, 
            edgecolors='w'
        )
    plt.legend()
    plt.title('Latent Space Visualization (First 2 PCA Components)')
else:
    plt.scatter(final_representation[:, 0], final_representation[:, 1], alpha=0.5)
    plt.title('Latent Space Visualization (First 2 PCA Components)')

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

## Next steps
- Use `outputs/autoencoder_pca_representation.csv` as input features for Quantum ML pipelines.
- Tune `encoding_dim`, `final_latent_dim`, network size, and training epochs for best downstream performance.

## Requirements
Run the following to install required packages if not already available:
```
pip install pandas numpy scikit-learn torch torchvision
```